In [ ]:
import os
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Functions

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

#### Output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/01_data_collection/{str_filename}'
df = pd.read_parquet(str_uri)
df['accountid'] = df['accountid'].astype(int)
# show
df

#### Drop targets

In [ ]:
list_cols = [col for col in df.columns if 'Early_Pay_Delinquency' in col]
list_cols.append('run_date')
df.drop(list_cols, axis=1, inplace=True)

#### Get targets - classification

In [ ]:
str_filename = 'df_targets.gzip'
str_uri = f's3://{str_project}/03_pull_targets/01_classification/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
df_tmp['bigAccountId'] = df_tmp['bigAccountId'].astype(int)
dict_rename = {
    'bigAccountId': 'accountid',
}
df_tmp.rename(columns=dict_rename, inplace=True)
df_tmp

#### Join

In [ ]:
df = pd.merge(
    left=df,
    right=df_tmp,
    on='accountid',
    how='left',
)
del df_tmp
df

#### Get targets - continuous

In [ ]:
str_filename = 'df_loss.gzip'
str_uri = f's3://{str_project}/03_pull_targets/02_regression/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=['bigAccountId','fltNetChgOff','months_on_books'],
)
df_tmp['bigAccountId'] = df_tmp['bigAccountId'].astype(int)
dict_rename = {
    'bigAccountId': 'accountid',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# show
df_tmp

#### Join

In [ ]:
list_int_months = [
    2,
    3,
    6,
    12,
    24,
]
for int_months in tqdm(list_int_months):
    df_tmp2 = df_tmp[df_tmp['months_on_books'] == int_months].copy()
    dict_rename = {
        'fltNetChgOff': f'fltNetChgOff_{int_months}',
    }
    df_tmp2.rename(columns=dict_rename, inplace=True)
    # join
    list_cols = [
        'accountid',
        f'fltNetChgOff_{int_months}',
    ]
    df = pd.merge(
        left=df,
        right=df_tmp2[list_cols],
        on='accountid',
        how='left',
    )
# save memory
del df_tmp
del df_tmp2

# show
df

#### Fillna

In [ ]:
list_cols = [col for col in df.columns if 'fltNetChgOff' in col]
df[list_cols] = df[list_cols].fillna(0)
# show
df

#### Charge-off severity

In [ ]:
# 60 days
df['co_at_60'] = df['fltNetChgOff_2'] / df['amtfinanced__app']
# 90 days
df['co_at_90'] = df['fltNetChgOff_3'] / df['amtfinanced__app']
# 180
df['co_at_180'] = df['fltNetChgOff_6'] / df['amtfinanced__app']
# 360
df['co_at_360'] = df['fltNetChgOff_12'] / df['amtfinanced__app']
# 720
df['co_at_720'] = df['fltNetChgOff_24'] / df['amtfinanced__app']
# show
df

#### Write to s3

In [ ]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)